# Neural Theorem Provers

## What This Is
Neural Theorem Provers (Rocktäschel & Riedel, 2017) perform logical inference in a differentiable way. Key insight: unification (matching logical terms) can be computed as vector similarity, enabling gradient-based learning of logic rules from data.

## How NTP Works
1. **Represent** facts and rules as vectors (embeddings)
2. **Prove** a query by template matching: try to unify the query with known facts or with the head of a rule
3. **Unification score**: instead of exact match, use similarity (dot product, cosine) between embeddings
4. **Chain rules**: proof of rule body + rule head unification = proof of conclusion
5. **Train**: maximize the proof score for positive examples, minimize for negatives

NTPs subsume embedding-based KG completion (TransE, RotatE) and add rule learning.

In [ ]:
import numpy as np
from typing import Dict, List, Tuple, Optional

np.random.seed(42)

# Simplified Neural Theorem Prover
# Knowledge base: set of (subject, predicate, object) triples
# Query: (subject, predicate, ?)
# Prove by finding chain of rules

class NeuralKB:
    """Neural knowledge base with embedding-based inference."""
    
    def __init__(self, embedding_dim: int = 16):
        self.dim = embedding_dim
        self.entity_embs: Dict[str, np.ndarray] = {}
        self.relation_embs: Dict[str, np.ndarray] = {}
        self.facts: List[Tuple[str, str, str]] = []
    
    def add_entity(self, name: str):
        if name not in self.entity_embs:
            self.entity_embs[name] = np.random.randn(self.dim) * 0.1
    
    def add_relation(self, name: str):
        if name not in self.relation_embs:
            self.relation_embs[name] = np.random.randn(self.dim) * 0.1
    
    def add_fact(self, s: str, p: str, o: str):
        self.add_entity(s); self.add_entity(o); self.add_relation(p)
        self.facts.append((s, p, o))
    
    def unification_score(self, v1: np.ndarray, v2: np.ndarray) -> float:
        """Cosine similarity as soft unification."""
        n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
        if n1 < 1e-8 or n2 < 1e-8: return 0.0
        return float(np.dot(v1, v2) / (n1 * n2))
    
    def forward_proof_score(self, s: str, p: str) -> List[Tuple[str, float]]:
        """Score all possible objects for query (s, p, ?)."""
        if s not in self.entity_embs or p not in self.relation_embs:
            return []
        
        # Direct evidence: find matching facts
        direct = [(o, 1.0) for fs, fp, o in self.facts if fs == s and fp == p]
        
        # Indirect: chain rules via embedding similarity
        # Find relations similar to p (possible rule heads)
        p_emb = self.relation_embs[p]
        related_preds = sorted(
            [(pred, self.unification_score(p_emb, emb))
             for pred, emb in self.relation_embs.items() if pred != p],
            key=lambda x: x[1], reverse=True
        )
        
        # Chain inference: find (s, related_pred, ?) facts
        indirect = []
        for rel, rel_sim in related_preds[:3]:
            for fs, fp, o in self.facts:
                if fs == s and fp == rel:
                    indirect.append((o, rel_sim * 0.8))
        
        all_scores = direct + indirect
        # Aggregate by object
        obj_scores = {}
        for o, sc in all_scores:
            obj_scores[o] = max(obj_scores.get(o, 0), sc)
        
        return sorted(obj_scores.items(), key=lambda x: x[1], reverse=True)

# Build a family knowledge graph
kb = NeuralKB(embedding_dim=8)

family_facts = [
    ('alice', 'parent_of', 'bob'),
    ('alice', 'parent_of', 'carol'),
    ('bob', 'parent_of', 'david'),
    ('carol', 'sibling_of', 'bob'),
    ('david', 'parent_of', 'eve'),
    ('bob', 'married_to', 'mary'),
]
for fact in family_facts:
    kb.add_fact(*fact)

print('Neural Knowledge Base:')
print(f'  Entities: {len(kb.entity_embs)}')
print(f'  Relations: {len(kb.relation_embs)}')
print(f'  Facts: {len(kb.facts)}')

print('\nQuery: who is bob a parent of?')
results = kb.forward_proof_score('bob', 'parent_of')
for obj, score in results[:5]:
    print(f'  bob parent_of {obj}: score={score:.3f}')

print('\nQuery: who is alice related to as grandparent?')
# Find alice -> parent_of -> X -> parent_of -> ?
alices_children = [o for s, p, o in kb.facts if s == 'alice' and p == 'parent_of']
grandchildren = []
for child in alices_children:
    for s, p, o in kb.facts:
        if s == child and p == 'parent_of':
            grandchildren.append(o)
print(f'  Alice grandchildren (2-hop): {grandchildren}')
